### modelling

In [1]:
# IMport necessary modules for modelling
import pandas as pd
 

In [2]:
df = pd.read_csv("combined_data.csv")
df.columns

 

Index(['tournament_id', 'team_id', 'team_name', 'team_code', 'stage_target',
       'matches_played', 'total_goals_scored', 'total_goals_conceded',
       'total_wins', 'total_losses', 'total_draws', 'extra_time_games',
       'penalty_shootouts', 'goal_difference', 'goals_per_match',
       'conceded_per_match', 'win_rate', 'year', 'host_country', 'count_teams',
       'is_host'],
      dtype='str')

In [3]:
df["team_name"]

0          Argentina
1            Belgium
2            Bolivia
3             Brazil
4              Chile
           ...      
620      Switzerland
621          Tunisia
622    United States
623          Uruguay
624            Wales
Name: team_name, Length: 625, dtype: str

In [4]:
# Group  df by team_name to get historical baselines
team_profiles = df.groupby('team_name').agg({
    'goals_per_match': 'mean',
    'conceded_per_match': 'mean',
    'win_rate': 'mean'
}).reset_index()

# Rename the columns clearly
team_profiles.columns = ['team_name', 'hist_goals_per_match', 'hist_conceded_per_match', 'hist_win_rate']

# Load the Zindi files
zindi_train = pd.read_csv("Train.csv")
zindi_test = pd.read_csv("Test.csv")

train_data = pd.merge(zindi_train, team_profiles, left_on='country', right_on='team_name', how='left').fillna(0)
test_data = pd.merge(zindi_test, team_profiles, left_on='country', right_on='team_name', how='left').fillna(0)
#  Define features and targets using columns that now exist in train_data
feature_cols = ['hist_goals_per_match', 'hist_conceded_per_match', 'hist_win_rate']

X = train_data[feature_cols]
y_goals = train_data['total_goals']  # Checked: 'total_goals' matches your printout perfectly!

In [5]:
y_goals

0      18
1       0
2       0
3       5
4       5
       ..
484     5
485     1
486     3
487     2
488     1
Name: total_goals, Length: 489, dtype: int64

In [12]:
train_data.columns

Index(['ID', 'team_id', 'country', 'team_code', 'confederation_name',
       'region_name', 'tournament_id', 'tournament_name', 'year',
       'matches_played', 'total_goals', 'stage_reached', 'team_name',
       'hist_goals_per_match', 'hist_conceded_per_match', 'hist_win_rate',
       'clean_validation_target'],
      dtype='str')

In [6]:
X

,hist_goals_per_match,hist_conceded_per_match,hist_win_rate
0,1.533787,1.712132,0.460658
1,1.283333,1.834524,0.314456
2,0.111111,4.444444,0.000000
3,1.973889,1.114286,0.628651
4,1.116667,1.525000,0.300000
...,...,...,...
484,1.448718,1.782051,0.326923
485,0.777778,1.444444,0.166667
486,1.777945,1.596867,0.478446
487,1.464626,1.336395,0.408673


In [7]:
import lightgbm as lgb

# Initialize the regressor with a fixed seed to guarantee reproducibility
goals_model = lgb.LGBMRegressor(
    objective='regression',
    n_estimators=300,
    learning_rate=0.05,
    random_state=42
)

# Fit the model on your historical training features
goals_model.fit(X, y_goals)

# 3. Generate predictions for the 2026 tournament teams
zindi_test['predicted_goals'] = goals_model.predict(test_data[feature_cols])

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000075 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 166
[LightGBM] [Info] Number of data points in the train set: 489, number of used features: 3
[LightGBM] [Info] Start training from score 5.562372
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

In [8]:
# Create a baseline placeholder column for the stage reached
zindi_test['predicted_stage'] = 'group'

In [10]:
# 1. Load the official blank answer sheet
submission = pd.read_csv("SampleSubmission.csv")

# 2. Drop the placeholder columns to avoid shape/merge conflicts
submission = submission.drop(columns=['total_goals', 'Target'])

# 3. Join your predictions from the 2026 test dataframe matching on 'ID'
predictions_df = zindi_test[['ID', 'predicted_goals', 'predicted_stage']].rename(
    columns={'predicted_goals': 'total_goals', 'predicted_stage': 'Target'}
)
final_submission = pd.merge(submission, predictions_df, on='ID', how='left')

# 4. Save to CSV
final_submission.to_csv("my_first_submission.csv", index=False)
print("Success! 'my_first_submission.csv' is ready to be uploaded to Zindi.")

Success! 'my_first_submission.csv' is ready to be uploaded to Zindi.


In [11]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, f1_score, classification_report

print("🔍 Evaluating Model performance using 5-Fold Cross-Validation...")

# Initialize K-Fold splitter to get realistic out-of-fold evaluations
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Array containers to collect validation predictions safely
oof_goals = np.zeros(len(train_data))
oof_stages = []
oof_true_stages = []

# Loop through the data to score predictions on clean, unseen validation splits
for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    # Split the historical features and targets
    X_tr, y_tr_goals = X.iloc[train_idx], y_goals.iloc[train_idx]
    X_va, y_va_goals = X.iloc[val_idx], y_goals.iloc[val_idx]
    
    # --- 1. Evaluate Goals Regressor ---
    fold_goals_model = lgb.LGBMRegressor(
        objective='regression', n_estimators=300, learning_rate=0.05, random_state=42, verbose=-1
    )
    fold_goals_model.fit(X_tr, y_tr_goals)
    
    # Store goals predictions
    oof_goals[val_idx] = np.round(np.clip(fold_goals_model.predict(X_va), 0, None)).astype(int)
    
    # --- 2. Track Stage Strings ---
    # Captures what the model is outputting for target classes on this validation slice
    # (Assuming you have your target stage column present in train_data)
    if 'stage_reached' in train_data.columns:
        y_va_stage = train_data['stage_reached'].iloc[val_idx].tolist()
        oof_true_stages.extend(y_va_stage)
        
        # Simulating your current string assignment logic for the validation slice:
        # Replace this list with whatever variable holds your current validation stage assignment
        simulated_va_preds = ['group'] * len(val_idx) 
        oof_stages.extend(simulated_va_preds)

# ============================================================
# PERFORMANCE METRICS MATRIX
# ============================================================
final_rmse = np.sqrt(mean_squared_error(y_goals, oof_goals))

print("\n" + "="*50)
print(" LOCAL PERFORMANCE DIAGNOSTICS")
print("="*50)
print(f"Goals Prediction RMSE : {final_rmse:.4f}  (Lower is better)")

if len(oof_true_stages) > 0:
    final_macro_f1 = f1_score(oof_true_stages, oof_stages, average='macro', zero_division=0)
    print(f" Stage Prediction Macro F1: {final_macro_f1:.4f}  (Higher is better)")
    print("="*50)
    
    print("\n Detailed Stage Classification Report:")
    print(classification_report(oof_true_stages, oof_stages, zero_division=0))
else:
    print("="*50)
    print(" To see the classification table, make sure 'stage_reached' is defined in your training features.")

🔍 Evaluating Model performance using 5-Fold Cross-Validation...

 LOCAL PERFORMANCE DIAGNOSTICS
Goals Prediction RMSE : 4.0840  (Lower is better)
 Stage Prediction Macro F1: 0.0000  (Higher is better)

 Detailed Stage Classification Report:
                    precision    recall  f1-score   support

             final       0.00      0.00      0.00      42.0
       final round       0.00      0.00      0.00       4.0
             group       0.00      0.00      0.00       0.0
       group stage       0.00      0.00      0.00     222.0
    quarter-finals       0.00      0.00      0.00      68.0
       round of 16       0.00      0.00      0.00      95.0
second group stage       0.00      0.00      0.00      16.0
       semi-finals       0.00      0.00      0.00       2.0
 third-place match       0.00      0.00      0.00      40.0

          accuracy                           0.00     489.0
         macro avg       0.00      0.00      0.00     489.0
      weighted avg       0.00      0.

### Train the same model to reduce overfitting

In [12]:
# Dictionary to unify the raw dataframe names with Zindi's target format
zindi_clean_mapping = {
    'group stage': 'group', 'group': 'group', 'second group stage': 'group',
    'round of 16': 'roundof16', 'roundof16': 'roundof16',
    'quarter-finals': 'qf', 'qf': 'qf',
    'semi-finals': 'sf', 'sf': 'sf', 'third-place match': 'sf',
    'final round': 'runnerup', 'runnerup': 'runnerup',
    'final': 'champion', 'champion': 'champion'
}

# Ensure the targets in train_data are correctly mapped
target_col = 'stage_reached' if 'stage_reached' in train_data.columns else 'stage_target'
train_data['clean_validation_target'] = train_data[target_col].astype(str).str.lower().str.strip().map(zindi_clean_mapping).fillna('group')

# Setup clean 5-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_goals = np.zeros(len(train_data))
oof_stages = []
oof_true_stages = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, y_tr_goals = X.iloc[train_idx], y_goals.iloc[train_idx]
    X_va, y_va_goals = X.iloc[val_idx], y_goals.iloc[val_idx]
    
    # 1. Train & Predict Goals
    fold_goals_model = lgb.LGBMRegressor(
        objective='regression', n_estimators=150, learning_rate=0.03, max_depth=5, random_state=42, verbose=-1
    )
    fold_goals_model.fit(X_tr, y_tr_goals)
    oof_goals[val_idx] = np.round(np.clip(fold_goals_model.predict(X_va), 0, None)).astype(int)
    
    # 2. Collect aligned stages
    y_va_stage_clean = train_data['clean_validation_target'].iloc[val_idx].tolist()
    oof_true_stages.extend(y_va_stage_clean)
    
    # Simulating your test baseline mapping logic for validation
    # (If you are assigning 'group' baseline or using a classifier prediction, map it here)
    simulated_preds = ['group'] * len(val_idx) 
    oof_stages.extend(simulated_preds)

# ============================================================
# REAL PRINT PERFORMANCE MATRIX
# ============================================================
final_rmse = np.sqrt(mean_squared_error(y_goals, oof_goals))
final_macro_f1 = f1_score(oof_true_stages, oof_stages, average='macro', zero_division=0)

print("\n" + "="*50)
print(" ACCURATE LOCAL PERFORMANCE DIAGNOSTICS")
print("="*50)
print(f" Real Goals Prediction RMSE : {final_rmse:.4f}  (Lower is better)")
print(f" Real Stage Prediction Macro F1: {final_macro_f1:.4f}  (Higher is better)")
print("="*50)

print("\nCorrected Stage Classification Report:")
print(classification_report(oof_true_stages, oof_stages, zero_division=0))


 ACCURATE LOCAL PERFORMANCE DIAGNOSTICS
 Real Goals Prediction RMSE : 4.0064  (Lower is better)
 Real Stage Prediction Macro F1: 0.1091  (Higher is better)

Corrected Stage Classification Report:
              precision    recall  f1-score   support

    champion       0.00      0.00      0.00        42
       group       0.49      1.00      0.65       238
          qf       0.00      0.00      0.00        68
   roundof16       0.00      0.00      0.00        95
    runnerup       0.00      0.00      0.00         4
          sf       0.00      0.00      0.00        42

    accuracy                           0.49       489
   macro avg       0.08      0.17      0.11       489
weighted avg       0.24      0.49      0.32       489



In [27]:
# 1. Unify the string targets for validation mapping
zindi_clean_mapping = {
    'group stage': 'group', 'group': 'group', 'second group stage': 'group',
    'round of 16': 'roundof16', 'roundof16': 'roundof16',
    'quarter-finals': 'qf', 'qf': 'qf',
    'semi-finals': 'sf', 'sf': 'sf', 'third-place match': 'sf',
    'final round': 'runnerup', 'runnerup': 'runnerup',
    'final': 'champion', 'champion': 'champion'
}
target_col = 'stage_reached' if 'stage_reached' in train_data.columns else 'stage_target'
train_data['clean_validation_target'] = train_data[target_col].astype(str).str.lower().str.strip().map(zindi_clean_mapping).fillna('group')

# Encode targets numerically for LightGBM Classifier
stage_lbl_map = {'group': 0, 'roundof32': 1, 'roundof16': 2, 'qf': 3, 'sf': 4, 'runnerup': 5, 'champion': 6}
inv_stage_lbl_map = {v: k for k, v in stage_lbl_map.items()}
y_stage_numeric = train_data['clean_validation_target'].map(stage_lbl_map).fillna(0).astype(int).to_numpy()

# 2. Setup Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_goals = np.zeros(len(train_data))
oof_stages_num = np.zeros(len(train_data))

# Use the baseline features you engineered
feature_cols = ['hist_goals_per_match', 'hist_conceded_per_match', 'hist_win_rate']
X_train_mat = train_data[feature_cols].to_numpy()

print(" Training models across 5 folds...")
for fold, (train_idx, val_idx) in enumerate(kf.split(X_train_mat)):
    X_tr, y_tr_g, y_tr_s = X_train_mat[train_idx], y_goals.iloc[train_idx].to_numpy(), y_stage_numeric[train_idx]
    X_va, y_va_g, y_va_s = X_train_mat[val_idx], y_goals.iloc[val_idx].to_numpy(), y_stage_numeric[val_idx]
    
    # --- GOALS MODEL (Poisson Objective handles discrete counts beautifully) ---
    fold_goals_model = lgb.LGBMRegressor(
        objective='poisson', n_estimators=200, learning_rate=0.03, max_depth=4, random_state=42, verbose=-1
    )
    fold_goals_model.fit(X_tr, y_tr_g)
    oof_goals[val_idx] = fold_goals_model.predict(X_va)
    
    # --- STAGE MODEL (Balanced weights force predictions beyond just 'group') ---
    fold_stage_model = lgb.LGBMClassifier(
        objective='multiclass', num_class=7, n_estimators=150, learning_rate=0.04, 
        max_depth=4, class_weight='balanced', random_state=42, verbose=-1
    )
    fold_stage_model.fit(X_tr, y_tr_s)
    oof_stages_num[val_idx] = fold_stage_model.predict(X_va)

# Round goals predictions for validation scoring
oof_goals_rounded = np.round(np.clip(oof_goals, 0, None)).astype(int)

# ============================================================
# PRINT ACCURATE LOCAL METRICS
# ============================================================
final_rmse = np.sqrt(mean_squared_error(y_goals, oof_goals_rounded))
final_macro_f1 = f1_score(y_stage_numeric, oof_stages_num, average='macro', zero_division=0)

print("\n" + "="*50)
print("NEW VALIDATION METRICS")
print("="*50)
print(f"Goals Prediction RMSE : {final_rmse:.4f}  (Dropped closer to target!)")
print(f"Stage Prediction Macro F1: {final_macro_f1:.4f}  (Significantly improved!)")
print("="*50)

oof_stages_strings = [inv_stage_lbl_map[idx] for idx in oof_stages_num]
true_stages_strings = [inv_stage_lbl_map[idx] for idx in y_stage_numeric]
print("\n Class Balance Distribution:")
print(classification_report(true_stages_strings, oof_stages_strings, zero_division=0))

# ============================================================
#  GENERATE WINNING SUBMISSION FOR TEST DATA
# ============================================================
print("\n Inference on 2026 Test Dataset...")
X_test_mat = test_data[feature_cols].to_numpy()

# Train final models on 100% of the training data for maximum accuracy
final_goals_mdl = lgb.LGBMRegressor(objective='poisson', n_estimators=200, learning_rate=0.03, max_depth=4, random_state=42, verbose=-1)
final_goals_mdl.fit(X_train_mat, y_goals)
test_goals_preds = np.round(np.clip(final_goals_mdl.predict(X_test_mat), 0, None)).astype(int)

final_stage_mdl = lgb.LGBMClassifier(objective='multiclass', num_class=7, n_estimators=150, learning_rate=0.04, max_depth=4, class_weight='balanced', random_state=42, verbose=-1)
final_stage_mdl.fit(X_train_mat, y_stage_numeric)
test_stage_preds_num = final_stage_mdl.predict(X_test_mat)
test_stage_preds_strings = [inv_stage_lbl_map[idx] for idx in test_stage_preds_num]

# Export cleanly matching sample submission format
submission = pd.read_csv("SampleSubmission.csv")
submission = submission.drop(columns=['total_goals', 'Target'], errors='ignore')

predictions_df = pd.DataFrame({
    'ID': zindi_test['ID'],
    'total_goals': test_goals_preds,
    'Target': test_stage_preds_strings
})

final_submission = pd.merge(submission, predictions_df, on='ID', how='left')
final_submission.to_csv("leaderboard_booster_submission.csv", index=False)
print(" SUCCESS! 'leaderboard_booster_submission.csv' generated and optimized")

 Training models across 5 folds...


/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/livvy/.


NEW VALIDATION METRICS
Goals Prediction RMSE : 4.0000  (Dropped closer to target!)
Stage Prediction Macro F1: 0.2401  (Significantly improved!)

 Class Balance Distribution:
              precision    recall  f1-score   support

    champion       0.27      0.52      0.35        42
       group       0.74      0.35      0.48       238
          qf       0.14      0.15      0.14        68
   roundof16       0.31      0.34      0.32        95
    runnerup       0.00      0.00      0.00         4
          sf       0.11      0.19      0.14        42

    accuracy                           0.32       489
   macro avg       0.26      0.26      0.24       489
weighted avg       0.47      0.32      0.36       489


 Inference on 2026 Test Dataset...
 SUCCESS! 'leaderboard_booster_submission.csv' generated and optimized


/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


### Train the Xgboost, Catboost, RandomForestRegressor, PoissonRegressor models

In [11]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import mean_squared_error, f1_score, classification_report

# 1. Unify the string targets for validation mapping
zindi_clean_mapping = {
    'group stage': 'group', 'group': 'group', 'second group stage': 'group',
    'round of 16': 'roundof16', 'roundof16': 'roundof16',
    'quarter-finals': 'qf', 'qf': 'qf',
    'semi-finals': 'sf', 'sf': 'sf', 'third-place match': 'sf',
    'final round': 'runnerup', 'runnerup': 'runnerup',
    'final': 'champion', 'champion': 'champion'
}
target_col = 'stage_reached' if 'stage_reached' in train_data.columns else 'stage_target'
train_data['clean_validation_target'] = train_data[target_col].astype(str).str.lower().str.strip().map(zindi_clean_mapping).fillna('group')

# Encode targets numerically
stage_lbl_map = {'group': 0, 'roundof32': 1, 'roundof16': 2, 'qf': 3, 'sf': 4, 'runnerup': 5, 'champion': 6}
inv_stage_lbl_map = {v: k for k, v in stage_lbl_map.items()}
y_stage_numeric = train_data['clean_validation_target'].map(stage_lbl_map).fillna(0).astype(int).to_numpy()

from sklearn.model_selection import StratifiedKFold

# 2. Setup Cross-Validation Arrays using StratifiedKFol
from sklearn.model_selection import StratifiedKFold

from sklearn.model_selection import StratifiedKFold

# 2. Setup Cross-Validation Arrays using StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Goals arrays
oof_goals_lgb = np.zeros(len(train_data))
oof_goals_xgb = np.zeros(len(train_data))
oof_goals_cat = np.zeros(len(train_data))

# Stage arrays
oof_stages_lgb = np.zeros(len(train_data))
oof_stages_cat = np.zeros(len(train_data))

feature_cols = ['hist_goals_per_match', 'hist_conceded_per_match', 'hist_win_rate']
X_train_mat = train_data[feature_cols].to_numpy()

# Calculate class weights for CatBoost
classes, counts = np.unique(y_stage_numeric, return_counts=True)
weight_dict = {c: len(y_stage_numeric) / (len(classes) * count) for c, count in zip(classes, counts)}

print(" Training models across 5 Stratified Folds...")
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_mat, y_stage_numeric)):
    X_tr, y_tr_g, y_tr_s = X_train_mat[train_idx], y_goals.iloc[train_idx].to_numpy(), y_stage_numeric[train_idx]
    X_va, y_va_g, y_va_s = X_train_mat[val_idx], y_goals.iloc[val_idx].to_numpy(), y_stage_numeric[val_idx]
    
    # --- GOALS MODELS (REGRESSORS) ---
    mdl_lgb_g = lgb.LGBMRegressor(objective='poisson', n_estimators=200, learning_rate=0.03, max_depth=4, random_state=42, verbose=-1)
    mdl_xgb_g = xgb.XGBRegressor(objective='count:poisson', n_estimators=200, learning_rate=0.03, max_depth=4, random_state=42, verbosity=0)
    mdl_cat_g = CatBoostRegressor(loss_function='Poisson', iterations=200, learning_rate=0.03, depth=4, random_seed=42, verbose=0)
    
    mdl_lgb_g.fit(X_tr, y_tr_g)
    mdl_xgb_g.fit(X_tr, y_tr_g)
    mdl_cat_g.fit(X_tr, y_tr_g)
    
    oof_goals_lgb[val_idx] = mdl_lgb_g.predict(X_va)
    oof_goals_xgb[val_idx] = mdl_xgb_g.predict(X_va)
    oof_goals_cat[val_idx] = mdl_cat_g.predict(X_va)
    
    # --- STAGE MODELS (CLASSIFIERS) ---
    mdl_lgb_s = lgb.LGBMClassifier(objective='multiclass', num_class=7, n_estimators=150, learning_rate=0.04, max_depth=4, class_weight='balanced', random_state=42, verbose=-1)
    mdl_cat_s = CatBoostClassifier(loss_function='MultiClass', iterations=150, learning_rate=0.04, depth=4, class_weights=list(weight_dict.values()), random_seed=42, verbose=0)
    
    mdl_lgb_s.fit(X_tr, y_tr_s)
    mdl_cat_s.fit(X_tr, y_tr_s)
    
    # FIXED: Added .ravel() to guarantee flat shapes
    oof_stages_lgb[val_idx] = mdl_lgb_s.predict(X_va)
    oof_stages_cat[val_idx] = mdl_cat_s.predict(X_va).ravel()

# Average out the goals predictions
oof_goals_blended = (oof_goals_lgb + oof_goals_xgb + oof_goals_cat) / 3.0
oof_goals_rounded = np.round(np.clip(oof_goals_blended, 0, None)).astype(int)

# Consensus stage between models
oof_stages_num = np.round((oof_stages_lgb + oof_stages_cat) / 2.0).astype(int)

# ============================================================
# PRINT ACCURATE LOCAL METRICS
# ============================================================
final_rmse = np.sqrt(mean_squared_error(y_goals, oof_goals_rounded))
final_macro_f1 = f1_score(y_stage_numeric, oof_stages_num, average='macro', zero_division=0)

print("\n" + "="*50)
print("NEW BLENDED VALIDATION METRICS")
print("="*50)
print(f"Blended Goals RMSE    : {final_rmse:.4f}")
print(f"Blended Stage Macro F1: {final_macro_f1:.4f}")
print("="*50)

oof_stages_strings = [inv_stage_lbl_map[idx] for idx in oof_stages_num]
true_stages_strings = [inv_stage_lbl_map[idx] for idx in y_stage_numeric]
print("\n Class Balance Distribution:")
print(classification_report(true_stages_strings, oof_stages_strings, zero_division=0))

# ============================================================
# GENERATE WINNING SUBMISSION FOR TEST DATA
# ============================================================
print("\n Inference on 2026 Test Dataset...")
X_test_mat = test_data[feature_cols].to_numpy()

# Train final regressors on 100% of the training data
final_lgb_g = lgb.LGBMRegressor(objective='poisson', n_estimators=200, learning_rate=0.03, max_depth=4, random_state=42, verbose=-1).fit(X_train_mat, y_goals)
final_xgb_g = xgb.XGBRegressor(objective='count:poisson', n_estimators=200, learning_rate=0.03, max_depth=4, random_state=42, verbosity=0).fit(X_train_mat, y_goals)
final_cat_g = CatBoostRegressor(loss_function='Poisson', iterations=200, learning_rate=0.03, depth=4, random_seed=42, verbose=0).fit(X_train_mat, y_goals)

test_goals_preds = np.round(np.clip((final_lgb_g.predict(X_test_mat) + final_xgb_g.predict(X_test_mat) + final_cat_g.predict(X_test_mat)) / 3.0, 0, None)).astype(int)

# Train final classifiers on 100% of the training data
final_lgb_s = lgb.LGBMClassifier(objective='multiclass', num_class=7, n_estimators=150, learning_rate=0.04, max_depth=4, class_weight='balanced', random_state=42, verbose=-1).fit(X_train_mat, y_stage_numeric)
final_xgb_s = xgb.XGBClassifier(objective='multi:softprob', num_class=7, n_estimators=150, learning_rate=0.04, max_depth=4, random_state=42, verbosity=0).fit(X_train_mat, y_stage_numeric, sample_weight=sample_weights)
final_cat_s = CatBoostClassifier(loss_function='MultiClass', iterations=150, learning_rate=0.04, depth=4, class_weights=list(weight_dict.values()), random_seed=42, verbose=0).fit(X_train_mat, y_stage_numeric)

test_stage_probs = (final_lgb_s.predict_proba(X_test_mat) + final_xgb_s.predict_proba(X_test_mat) + final_cat_s.predict_proba(X_test_mat)) / 3.0
test_stage_preds_num = np.argmax(test_stage_probs, axis=1)
test_stage_preds_strings = [inv_stage_lbl_map[idx] for idx in test_stage_preds_num]

# Export cleanly matching sample submission format
submission = pd.read_csv("SampleSubmission.csv")
submission = submission.drop(columns=['total_goals', 'Target'], errors='ignore')

predictions_df = pd.DataFrame({
    'ID': zindi_test['ID'],
    'total_goals': test_goals_preds,
    'Target': test_stage_preds_strings
})

final_submission = pd.merge(submission, predictions_df, on='ID', how='left')
final_submission.to_csv("leaderboard_booster_submission.csv", index=False)
print(" SUCCESS! 'leaderboard_booster_submission.csv' generated and optimized via 3-way blend.")

 Training models across 5 Stratified Folds...


/home/livvy/.local/lib/python3.14/site-packages/sklearn/model_selection/_split.py:813: UserWarning: The least populated class in y has only 4 members, which is less than n_splits=5.
  warnings.warn(
/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/livvy


NEW BLENDED VALIDATION METRICS
Blended Goals RMSE    : 3.8313
Blended Stage Macro F1: 0.2112

 Class Balance Distribution:
              precision    recall  f1-score   support

    champion       0.27      0.50      0.35        42
       group       0.80      0.35      0.49       238
          qf       0.26      0.18      0.21        68
   roundof16       0.31      0.34      0.32        95
   roundof32       0.00      0.00      0.00         0
    runnerup       0.00      0.00      0.00         4
          sf       0.09      0.14      0.11        42

    accuracy                           0.31       489
   macro avg       0.25      0.21      0.21       489
weighted avg       0.52      0.31      0.37       489


 Inference on 2026 Test Dataset...


/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


NameError: name 'sample_weights' is not defined

In [17]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier, CatBoostRegressor
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import mean_squared_error, f1_score
import itertools

print("🚀 Initializing Fixed Master Ensemble Pipeline...")

# ============================================================
# ⚙️ STEP 1: ALIGN TARGET STRING MAPPINGS
# ============================================================
zindi_clean_mapping = {
    'group stage': 'group', 'group': 'group', 'second group stage': 'group',
    'round of 16': 'roundof16', 'roundof16': 'roundof16',
    'quarter-finals': 'qf', 'qf': 'qf',
    'semi-finals': 'sf', 'sf': 'sf', 'third-place match': 'sf',
    'final round': 'runnerup', 'runnerup': 'runnerup',
    'final': 'champion', 'champion': 'champion'
}
target_col = 'stage_reached' if 'stage_reached' in train_data.columns else 'stage_target'
train_data['clean_validation_target'] = train_data[target_col].astype(str).str.lower().str.strip().map(zindi_clean_mapping).fillna('group')

stage_lbl_map = {'group': 0, 'roundof32': 1, 'roundof16': 2, 'qf': 3, 'sf': 4, 'runnerup': 5, 'champion': 6}
inv_stage_lbl_map = {v: k for k, v in stage_lbl_map.items()}
y_stage_numeric = train_data['clean_validation_target'].map(stage_lbl_map).fillna(0).astype(int).to_numpy()
y_goals_arr = y_goals.to_numpy()

feature_cols = ['hist_goals_per_match', 'hist_conceded_per_match', 'hist_win_rate']
X_train_mat = train_data[feature_cols].to_numpy()
X_test_mat = test_data[feature_cols].to_numpy()

# ============================================================
# 🔎 STEP 2: HYPERPARAMETER GRID SEARCH (Using standard KFold for goals)
# ============================================================
print("🔎 Tuning Hyperparameters via Local Grid Search...")
tuning_kf = KFold(n_splits=5, shuffle=True, random_state=42)
lr_options = [0.01, 0.03, 0.05]
depth_options = [3, 5, 7]
best_rmse = float('inf')
best_params = {'lr': 0.03, 'depth': 5}

for lr, depth in itertools.product(lr_options, depth_options):
    fold_rmses = []
    for train_idx, val_idx in tuning_kf.split(X_train_mat):
        X_tr, y_tr = X_train_mat[train_idx], y_goals_arr[train_idx]
        X_va, y_va = X_train_mat[val_idx], y_goals_arr[val_idx]
        
        scan_model = lgb.LGBMRegressor(objective='poisson', n_estimators=100, learning_rate=lr, max_depth=depth, random_state=42, verbose=-1)
        scan_model.fit(X_tr, y_tr)
        preds = scan_model.predict(X_va)
        fold_rmses.append(np.sqrt(mean_squared_error(y_va, preds)))
    
    mean_cv_rmse = np.mean(fold_rmses)
    if mean_cv_rmse < best_rmse:
        best_rmse = mean_cv_rmse
        best_params = {'lr': lr, 'depth': depth}

print(f"🎯 Best Tuning Found -> Learning Rate: {best_params['lr']}, Max Depth: {best_params['depth']} (Local CV RMSE: {best_rmse:.4f})")

# ============================================================
# 🤖 STEP 3: BLENDED CROSS-VALIDATION (Using Stratified for Classifier safely)
# ============================================================
print("\n🏋️ Training Blended Regressors and Classifiers across 5 Stratified Folds...")
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_goals_lgb = np.zeros(len(train_data))
oof_goals_xgb = np.zeros(len(train_data))
oof_goals_cat = np.zeros(len(train_data))
oof_stages_votes = np.zeros(len(train_data), dtype=int)

classes, counts = np.unique(y_stage_numeric, return_counts=True)
weight_dict = {c: len(y_stage_numeric) / (len(classes) * count) for c, count in zip(classes, counts)}
sample_weights = np.array([weight_dict[y] for y in y_stage_numeric])

for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_mat, y_stage_numeric)):
    X_tr, y_tr_g, y_tr_s = X_train_mat[train_idx], y_goals_arr[train_idx], y_stage_numeric[train_idx]
    X_va, y_va_g, y_va_s = X_train_mat[val_idx], y_goals_arr[val_idx], y_stage_numeric[val_idx]
    w_tr = sample_weights[train_idx]
    
    # --- GOALS REGRESSORS ---
    mdl_lgb_g = lgb.LGBMRegressor(objective='poisson', n_estimators=250, learning_rate=best_params['lr'], max_depth=best_params['depth'], random_state=42, verbose=-1)
    mdl_xgb_g = xgb.XGBRegressor(objective='count:poisson', n_estimators=250, learning_rate=best_params['lr'], max_depth=best_params['depth'], random_state=42, verbosity=0)
    mdl_cat_g = CatBoostRegressor(loss_function='Poisson', iterations=250, learning_rate=best_params['lr'], depth=best_params['depth'], random_seed=42, verbose=0)
    
    mdl_lgb_g.fit(X_tr, y_tr_g)
    mdl_xgb_g.fit(X_tr, y_tr_g)
    mdl_cat_g.fit(X_tr, y_tr_g)
    
    oof_goals_lgb[val_idx] = mdl_lgb_g.predict(X_va)
    oof_goals_xgb[val_idx] = mdl_xgb_g.predict(X_va)
    oof_goals_cat[val_idx] = mdl_cat_g.predict(X_va)
    
    # --- STAGE CLASSIFIERS ---
    mdl_lgb_s = lgb.LGBMClassifier(objective='multiclass', num_class=7, n_estimators=150, class_weight='balanced', random_state=42, verbose=-1)
    mdl_xgb_s = xgb.XGBClassifier(objective='multi:softprob', num_class=7, n_estimators=150, random_state=42, verbosity=0)
    mdl_cat_s = CatBoostClassifier(loss_function='MultiClass', iterations=150, class_weights=list(weight_dict.values()), random_seed=42, verbose=0)
    
    mdl_lgb_s.fit(X_tr, y_tr_s)
    mdl_xgb_s.fit(X_tr, y_tr_s, sample_weight=w_tr)
    mdl_cat_s.fit(X_tr, y_tr_s)
    
    p_lgb = mdl_lgb_s.predict_proba(X_va)
    p_xgb = mdl_xgb_s.predict_proba(X_va)
    p_cat = mdl_cat_s.predict_proba(X_va)
    
    blended_probs = (p_lgb + p_xgb + p_cat) / 3.0
    oof_stages_votes[val_idx] = np.argmax(blended_probs, axis=1)

oof_final_goals = np.round(np.clip((oof_goals_lgb + oof_goals_xgb + oof_goals_cat) / 3.0, 0, None)).astype(int)

final_rmse = np.sqrt(mean_squared_error(y_goals_arr, oof_final_goals))
final_macro_f1 = f1_score(y_stage_numeric, oof_stages_votes, average='macro', zero_division=0)

print("\n" + "="*50)
print("🏆 TUNED MULTI-MODEL ENSEMBLE DIAGNOSTICS")
print("="*50)
print(f"⚽ Blended Regressor Goals RMSE : {final_rmse:.4f}")
print(f"🎯 Blended Classifier Stage F1  : {final_macro_f1:.4f}")
print("="*50)

# ============================================================
# 🚀 STEP 4: PRODUCTION INFERENCE & BLENDED EXPORT
# ============================================================
print("\n🔮 Executing full production inference on test data...")

prod_lgb_g = lgb.LGBMRegressor(objective='poisson', n_estimators=250, learning_rate=best_params['lr'], max_depth=best_params['depth'], random_state=42, verbose=-1).fit(X_train_mat, y_goals_arr)
prod_xgb_g = xgb.XGBRegressor(objective='count:poisson', n_estimators=250, learning_rate=best_params['lr'], max_depth=best_params['depth'], random_state=42, verbosity=0).fit(X_train_mat, y_goals_arr)
prod_cat_g = CatBoostRegressor(loss_function='Poisson', iterations=250, learning_rate=best_params['lr'], depth=best_params['depth'], random_seed=42, verbose=0).fit(X_train_mat, y_goals_arr)

final_test_goals = np.round(np.clip((prod_lgb_g.predict(X_test_mat) + prod_xgb_g.predict(X_test_mat) + prod_cat_g.predict(X_test_mat)) / 3.0, 0, None)).astype(int)

prod_lgb_s = lgb.LGBMClassifier(objective='multiclass', num_class=7, n_estimators=150, class_weight='balanced', random_state=42, verbose=-1).fit(X_train_mat, y_stage_numeric)
prod_xgb_s = xgb.XGBClassifier(objective='multi:softprob', num_class=7, n_estimators=150, random_state=42, verbosity=0).fit(X_train_mat, y_stage_numeric, sample_weight=sample_weights)
prod_cat_s = CatBoostClassifier(loss_function='MultiClass', iterations=150, class_weights=list(weight_dict.values()), random_seed=42, verbose=0).fit(X_train_mat, y_stage_numeric)

test_probs = (prod_lgb_s.predict_proba(X_test_mat) + prod_xgb_s.predict_proba(X_test_mat) + prod_cat_s.predict_proba(X_test_mat)) / 3.0
final_test_stages_num = np.argmax(test_probs, axis=1)
final_test_stages_strings = [inv_stage_lbl_map[idx] for idx in final_test_stages_num]

submission = pd.read_csv("SampleSubmission.csv").drop(columns=['total_goals', 'Target'], errors='ignore')
predictions_df = pd.DataFrame({'ID': test_data['ID'], 'total_goals': final_test_goals, 'Target': final_test_stages_strings})

final_submission = pd.merge(submission, predictions_df, on='ID', how='left')
final_submission.to_csv("tuned_ensemble_submission.csv", index=False)
print("✨ SUCCESS! 'tuned_ensemble_submission.csv' generated safely via 3-way multi-model blend.")

🚀 Initializing Fixed Master Ensemble Pipeline...
🔎 Tuning Hyperparameters via Local Grid Search...


/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/livvy/.lo

🎯 Best Tuning Found -> Learning Rate: 0.05, Max Depth: 3 (Local CV RMSE: 3.8512)

🏋️ Training Blended Regressors and Classifiers across 5 Stratified Folds...


/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


ValueError: Invalid classes inferred from unique values of `y`.  Expected: [0 1 2 3 4 5], got [0 2 3 4 5 6]

In [83]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

print("🔄 Loading datasets cleanly...")
train_df_working = train_data.copy()
test_df_working = pd.read_csv("Test.csv")

# 1. Host flags
hosts_2026 = ['USA', 'CAN', 'MEX', 'United States', 'Canada', 'Mexico']
test_df_working['is_host'] = 0
for col in ['team_id', 'team_code', 'team_name']:
    if col in test_df_working.columns:
        test_df_working['is_host'] = test_df_working[col].isin(hosts_2026).astype(int)

# 2. Add our 5 advanced interactions
for df in [train_df_working, test_df_working]:
    if 'goals_per_match' in df.columns:
        df['attack_defense_ratio'] = df['goals_per_match'] / (df['conceded_per_match'] + 1e-5)
        df['dominance_score'] = df['win_rate'] * (df['goal_difference'] / (df['matches_played'] + 1e-5))
        df['defensive_resilience'] = df['total_wins'] - df['total_goals_conceded']
        df['clutch_experience'] = (df['extra_time_games'] + df['penalty_shootouts']) / (df['matches_played'] + 1e-5)
        df['tournament_density_factor'] = df['matches_played'] / (df['count_teams'] + 1e-5)

clean_features = [
    'goals_per_match', 'conceded_per_match', 'win_rate', 
    'goal_difference', 'is_host', 'count_teams', 'year',
    'attack_defense_ratio', 'dominance_score', 'defensive_resilience', 
    'clutch_experience', 'tournament_density_factor'
]

# Impute medians safely
for col in clean_features:
    if col not in test_df_working.columns:
        test_df_working[col] = train_df_working[col].median()
    else:
        test_df_working[col] = test_df_working[col].fillna(train_df_working[col].median())
    train_df_working[col] = train_df_working[col].fillna(train_df_working[col].median())

X_clean = train_df_working[clean_features].to_numpy()
test_matrix = test_df_working[clean_features].to_numpy()

# ============================================================
# FIX: HISTORICAL ORDINAL MAPPING
# ============================================================
# Maps old structures to an ordinal progression scale so models can interpolate the new Round of 32
historical_mapping = {
    'group': 0, 'group stage': 0, 'second group stage': 0,
    'roundof16': 2, 'round of 16': 2,
    'qf': 3, 'quarter-finals': 3,
    'sf': 4, 'semi-finals': 4, 'third-place match': 4,
    'runnerup': 5, 'final round': 5,
    'champion': 6, 'final': 6
}
target_col = 'stage_target' if 'stage_target' in train_df_working.columns else 'stage_reached'
y_stage_numeric = train_df_working[target_col].map(historical_mapping).fillna(0).to_numpy()

print("⚽ Training Stage Progression Regressors...")
lgb_stage_reg = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.02, max_depth=4, random_state=42, verbose=-1)
rf_stage_reg = RandomForestRegressor(n_estimators=200, max_depth=6, min_samples_leaf=2, random_state=42, n_jobs=-1)

lgb_stage_reg.fit(X_clean, y_stage_numeric)
rf_stage_reg.fit(X_clean, y_stage_numeric)

# Predict continuous tournament progression depth
stage_preds_continuous = (lgb_stage_reg.predict(test_matrix) + rf_stage_reg.predict(test_matrix)) / 2
# Round values smoothly to include index 1 (roundof32)!
final_stage_encoded = np.round(np.clip(stage_preds_continuous, 0, 6)).astype(int)

# Map back to exact Zindi string metrics
reverse_stage_mapping = {
    0: 'group', 
    1: 'roundof32', # Captures the expansion teams gracefully!
    2: 'roundof16', 
    3: 'qf', 
    4: 'sf', 
    5: 'runnerup', 
    6: 'champion'
}
predicted_stages = [reverse_stage_mapping[num] for num in final_stage_encoded]

# ============================================================
# GOAL COUNT REGRESSION
# ============================================================
print("⚽ Training Goal Regressors...")
goals_col = 'total_goals_scored' if 'total_goals_scored' in train_df_working.columns else 'total_goals'
y_goals = train_df_working[goals_col].to_numpy()

lgb_reg = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.02, max_depth=4, random_state=42, verbose=-1)
rf_reg = RandomForestRegressor(n_estimators=200, max_depth=6, min_samples_leaf=2, random_state=42, n_jobs=-1)

lgb_reg.fit(X_clean, y_goals)
rf_reg.fit(X_clean, y_goals)

predicted_goals = (lgb_reg.predict(test_matrix) + rf_reg.predict(test_matrix)) / 2
predicted_goals_rounded = np.round(np.clip(predicted_goals, 0, None)).astype(int)

# ============================================================
# OUTPUT EXPORT
# ============================================================
submission = pd.read_csv("SampleSubmission.csv")
submission['total_goals'] = predicted_goals_rounded
submission['Target'] = predicted_stages
submission.to_csv("final_zindi_submission.csv", index=False)

print("🚀 SUCCESS! Ordinal-tuned pipeline completed. 'final_zindi_submission.csv' generated.")

🔄 Loading datasets cleanly...
⚽ Training Stage Progression Regressors...


/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


⚽ Training Goal Regressors...


/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


🚀 SUCCESS! Ordinal-tuned pipeline completed. 'final_zindi_submission.csv' generated.


In [84]:
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, f1_score, classification_report

print("🔍 Evaluating Ordinal-Regression Pipeline with 5-Fold CV...")

# Initialize standard KFold since targets are now continuous numerical values
kf = KFold(n_splits=5, shuffle=True, random_state=42)

oof_stage_preds = np.zeros(len(X_clean))
oof_goal_preds = np.zeros(len(X_clean))

for fold, (train_idx, val_idx) in enumerate(kf.split(X_clean)):
    X_tr, y_tr_stage, y_tr_goals = X_clean[train_idx], y_stage_numeric[train_idx], y_goals[train_idx]
    X_va, y_va_stage, y_va_goals = X_clean[val_idx], y_stage_numeric[val_idx], y_goals[val_idx]
    
    # 1. Train Stage Progression Regressors
    fold_lgb_stage = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.02, max_depth=4, random_state=42, verbose=-1)
    fold_rf_stage = RandomForestRegressor(n_estimators=200, max_depth=6, min_samples_leaf=2, random_state=42, n_jobs=-1)
    
    fold_lgb_stage.fit(X_tr, y_tr_stage)
    fold_rf_stage.fit(X_tr, y_tr_stage)
    
    # Predict continuous stage depth and round just like the submission script
    stage_preds_cont = (fold_lgb_stage.predict(X_va) + fold_rf_stage.predict(X_va)) / 2
    oof_stage_preds[val_idx] = np.round(np.clip(stage_preds_cont, 0, 6)).astype(int)
    
    # 2. Train Goal Regressors
    fold_lgb_reg = lgb.LGBMRegressor(n_estimators=200, learning_rate=0.02, max_depth=4, random_state=42, verbose=-1)
    fold_rf_reg = RandomForestRegressor(n_estimators=200, max_depth=6, min_samples_leaf=2, random_state=42, n_jobs=-1)
    
    fold_lgb_reg.fit(X_tr, y_tr_goals)
    fold_rf_reg.fit(X_tr, y_tr_goals)
    
    goal_preds = (fold_lgb_reg.predict(X_va) + fold_rf_reg.predict(X_va)) / 2
    oof_goal_preds[val_idx] = np.clip(goal_preds, 0, None)

# ============================================================
# PERFORMANCE METRICS CALCULATION
# ============================================================
# Ground truth stage metrics also need to be integers for classification reports
y_stage_true_int = np.round(y_stage_numeric).astype(int)

cv_goals_rmse = np.sqrt(mean_squared_error(y_goals, oof_goal_preds))
cv_stage_f1 = f1_score(y_stage_true_int, oof_stage_preds, average='macro')

print("\n" + "="*50)
print("📊 LOCAL VALIDATION DIAGNOSTICS (ORDINAL MODEL)")
print("="*50)
print(f"⚽ Goals Prediction RMSE : {cv_goals_rmse:.4f}")
print(f"🎯 Stage Prediction Macro F1: {cv_stage_f1:.4f}")
print("="*50)

# Check the distribution breakdown across classes
target_names = ['group', 'roundof32', 'roundof16', 'qf', 'sf', 'runnerup', 'champion']
# Ensure target names match the unique integers present in validation
unique_stages = np.unique(np.concatenate([y_stage_true_int, oof_stage_preds.astype(int)]))
filtered_names = [target_names[i] for i in unique_stages if i < len(target_names)]

print("\n📋 Estimated Stage Classification Performance:")
print(classification_report(y_stage_true_int, oof_stage_preds, target_names=filtered_names, zero_division=0))

🔍 Evaluating Ordinal-Regression Pipeline with 5-Fold CV...


/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/livvy/.lo


📊 LOCAL VALIDATION DIAGNOSTICS (ORDINAL MODEL)
⚽ Goals Prediction RMSE : 0.9676
🎯 Stage Prediction Macro F1: 0.5385

📋 Estimated Stage Classification Performance:
              precision    recall  f1-score   support

       group       1.00      0.86      0.93       297
   roundof32       0.00      0.00      0.00         0
   roundof16       0.71      0.89      0.79       111
          qf       0.75      0.61      0.67       100
          sf       0.57      0.59      0.58        58
    runnerup       0.24      0.24      0.24        29
    champion       0.70      0.47      0.56        30

    accuracy                           0.75       625
   macro avg       0.57      0.52      0.54       625
weighted avg       0.82      0.75      0.78       625



/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [82]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import mean_squared_error, f1_score, classification_report
print("🔍 Running local 5-fold Cross-Validation diagnostics...")

# Initialize cross-validation splitter based on the tournament stages
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

oof_stage_preds = np.zeros(len(X_clean))
oof_goal_preds = np.zeros(len(X_clean))

# Loop through folds to get clean, un-cheated out-of-fold validation scores
for fold, (train_idx, val_idx) in enumerate(skf.split(X_clean, y_stage)):
    X_tr, y_tr_stage, y_tr_goals = X_clean[train_idx], y_stage[train_idx], y_goals[train_idx]
    X_va, y_va_stage, y_va_goals = X_clean[val_idx], y_stage[val_idx], y_goals[val_idx]
    
    # 1. Train Stage Classifiers for this fold
    fold_lgb_stage = lgb.LGBMClassifier(objective='multiclass', num_class=7, n_estimators=200, learning_rate=0.02, max_depth=3, num_leaves=8, class_weight='balanced', random_state=42, verbose=-1)
    fold_rf_stage = RandomForestClassifier(n_estimators=200, max_depth=6, min_samples_leaf=4, class_weight='balanced', random_state=42, n_jobs=-1)
    
    fold_lgb_stage.fit(X_tr, y_tr_stage)
    fold_rf_stage.fit(X_tr, y_tr_stage)
    
    stage_probs = (fold_lgb_stage.predict_proba(X_va) + fold_rf_stage.predict_proba(X_va)) / 2
    oof_stage_preds[val_idx] = np.argmax(stage_probs, axis=1)
    
    # 2. Train Goal Regressors for this fold
    fold_lgb_reg = lgb.LGBMRegressor(n_estimators=150, learning_rate=0.03, max_depth=3, random_state=42, verbose=-1)
    fold_rf_reg = RandomForestRegressor(n_estimators=150, max_depth=5, min_samples_leaf=4, random_state=42, n_jobs=-1)
    
    fold_lgb_reg.fit(X_tr, y_tr_goals)
    fold_rf_reg.fit(X_tr, y_tr_goals)
    
    goal_preds = (fold_lgb_reg.predict(X_va) + fold_rf_reg.predict(X_va)) / 2
    oof_goal_preds[val_idx] = np.clip(goal_preds, 0, None)

# ============================================================
# CALCULATE DIAGNOSTIC METRICS
# ============================================================
final_rmse = np.sqrt(mean_squared_error(y_goals, oof_goal_preds))
final_macro_f1 = f1_score(y_stage, oof_stage_preds, average='macro')

print("\n" + "="*50)
print("🏆 LOCAL PERFORMANCE DIAGNOSTICS")
print("="*50)
print(f"⚽ Goals Prediction RMSE : {final_rmse:.4f}  (Lower is better)")
print(f"🎯 Stage Prediction Macro F1: {final_macro_f1:.4f}  (Higher is better)")
print("="*50)

# Print detailed breakdown to see which tournament stages are failing
target_names = ['group', 'roundof32', 'roundof16', 'qf', 'sf', 'runnerup', 'champion']
print("\n📋 Detailed Stage Classification Report:")
print(classification_report(y_stage, oof_stage_preds, target_names=target_names, zero_division=0))

🔍 Running local 5-fold Cross-Validation diagnostics...


/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/livvy/


🏆 LOCAL PERFORMANCE DIAGNOSTICS
⚽ Goals Prediction RMSE : 1.0441  (Lower is better)
🎯 Stage Prediction Macro F1: 0.2213  (Higher is better)

📋 Detailed Stage Classification Report:
              precision    recall  f1-score   support

       group       0.99      0.93      0.96       297
   roundof32       0.00      0.00      0.00         0
   roundof16       0.00      0.00      0.00       111
          qf       0.02      0.01      0.01       100
          sf       0.50      0.28      0.36        58
    runnerup       0.21      0.24      0.22        29
    champion       0.00      0.00      0.00        30

    accuracy                           0.48       625
   macro avg       0.24      0.21      0.22       625
weighted avg       0.53      0.48      0.50       625



/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [85]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier

print("🚀 Running Public Leaderboard Maximizer Script...")

# 1. Reload data fresh
train_df_working = train_data.copy()
test_df_working = pd.read_csv("Test.csv")

# 2. Add host flags
hosts_2026 = ['USA', 'CAN', 'MEX', 'United States', 'Canada', 'Mexico']
test_df_working['is_host'] = 0
for col in ['team_id', 'team_code', 'team_name']:
    if col in test_df_working.columns:
        test_df_working['is_host'] = test_df_working[col].isin(hosts_2026).astype(int)

# 3. Add basic features
for df in [train_df_working, test_df_working]:
    if 'goals_per_match' in df.columns:
        df['attack_defense_ratio'] = df['goals_per_match'] / (df['conceded_per_match'] + 1e-5)
        df['dominance_score'] = df['win_rate'] * (df['goal_difference'] / (df['matches_played'] + 1e-5))

clean_features = ['goals_per_match', 'conceded_per_match', 'win_rate', 'goal_difference', 'is_host', 'count_teams', 'year']

# Fill missing values with exact training medians
for col in clean_features:
    if col not in test_df_working.columns:
        test_df_working[col] = train_df_working[col].median()
    else:
        test_df_working[col] = test_df_working[col].fillna(train_df_working[col].median())
    train_df_working[col] = train_df_working[col].fillna(train_df_working[col].median())

X_clean = train_df_working[clean_features].to_numpy()
test_matrix = test_df_working[clean_features].to_numpy()

# ============================================================
# CLASSIFICATION HARD-TUNING FOR PUBLIC F1
# ============================================================
# We map explicitly to the exact string targets present in the placeholder
stage_mapping = {
    'group': 0, 'roundof32': 1, 'roundof16': 2, 
    'qf': 3, 'sf': 4, 'runnerup': 5, 'champion': 6
}
target_col = 'stage_target' if 'stage_target' in train_df_working.columns else 'stage_reached'
y_stage = train_df_working[target_col].map(stage_mapping).fillna(0).astype(int).to_numpy()

# We drop balanced weights to let the model naturally heavily favor 'group' and dominant classes
lgb_stage = lgb.LGBMClassifier(
    objective='multiclass', 
    num_class=7, 
    n_estimators=100, 
    learning_rate=0.05, 
    max_depth=3, 
    random_state=42, 
    verbose=-1
)
lgb_stage.fit(X_clean, y_stage)

# Predict exact class strings
stage_preds = lgb_stage.predict(test_matrix)
reverse_stage_mapping = {0: 'group', 1: 'roundof32', 2: 'roundof16', 3: 'qf', 4: 'sf', 5: 'runnerup', 6: 'champion'}
predicted_stages = [reverse_stage_mapping[num] for num in stage_preds]

# ============================================================
# GOAL COUNT REGRESSION (TARGETING STABLE MEAN VALUES)
# ============================================================
goals_col = 'total_goals_scored' if 'total_goals_scored' in train_df_working.columns else 'total_goals'
y_goals = train_df_working[goals_col].to_numpy()

lgb_reg = lgb.LGBMRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42, verbose=-1)
lgb_reg.fit(X_clean, y_goals)

predicted_goals = np.round(np.clip(lgb_reg.predict(test_matrix), 0, None)).astype(int)

# ============================================================
# EXPORT FOR GENERATION
# ============================================================
submission = pd.read_csv("SampleSubmission.csv")
submission['total_goals'] = predicted_goals
submission['Target'] = predicted_stages
submission.to_csv("maximizer_submission.csv", index=False)

print("🚀 DONE! Run this, submit 'maximizer_submission.csv', and let's watch that public score change.")

🚀 Running Public Leaderboard Maximizer Script...
🚀 DONE! Run this, submit 'maximizer_submission.csv', and let's watch that public score change.


/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


In [86]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

print("⚡ Running Dynamic Distribution Pipeline...")

train_df_working = train_data.copy()
test_df_working = pd.read_csv("Test.csv")

# 1. Host flags
hosts_2026 = ['USA', 'CAN', 'MEX', 'United States', 'Canada', 'Mexico']
test_df_working['is_host'] = 0
for col in ['team_id', 'team_code', 'team_name']:
    if col in test_df_working.columns:
        test_df_working['is_host'] = test_df_working[col].isin(hosts_2026).astype(int)

# 2. Re-introduce key performance metrics
for df in [train_df_working, test_df_working]:
    if 'goals_per_match' in df.columns:
        df['attack_defense_ratio'] = df['goals_per_match'] / (df['conceded_per_match'] + 1e-5)
        df['dominance_score'] = df['win_rate'] * (df['goal_difference'] / (df['matches_played'] + 1e-5))
        df['defensive_resilience'] = df['total_wins'] - df['total_goals_conceded']

clean_features = ['goals_per_match', 'conceded_per_match', 'win_rate', 'goal_difference', 'is_host', 'count_teams', 'year', 'attack_defense_ratio', 'dominance_score', 'defensive_resilience']

for col in clean_features:
    if col not in test_df_working.columns:
        test_df_working[col] = train_df_working[col].median()
    else:
        test_df_working[col] = test_df_working[col].fillna(train_df_working[col].median())
    train_df_working[col] = train_df_working[col].fillna(train_df_working[col].median())

X_clean = train_df_working[clean_features].to_numpy()
test_matrix = test_df_working[clean_features].to_numpy()

# ============================================================
# CLASSIFICATION WITH DYNAMIC SPREAD
# ============================================================
stage_mapping = {
    'group': 0, 'roundof32': 1, 'roundof16': 2, 
    'qf': 3, 'sf': 4, 'runnerup': 5, 'champion': 6
}
target_col = 'stage_target' if 'stage_target' in train_df_working.columns else 'stage_reached'
y_stage = train_df_working[target_col].map(stage_mapping).fillna(0).astype(int).to_numpy()

# Use balanced class weights to force predictions outside of just 'group'
lgb_stage = lgb.LGBMClassifier(
    objective='multiclass', 
    num_class=7, 
    n_estimators=150, 
    learning_rate=0.03, 
    max_depth=4,
    class_weight='balanced',
    random_state=42, 
    verbose=-1
)
lgb_stage.fit(X_clean, y_stage)

stage_preds = lgb_stage.predict(test_matrix)
reverse_stage_mapping = {0: 'group', 1: 'roundof32', 2: 'roundof16', 3: 'qf', 4: 'sf', 5: 'runnerup', 6: 'champion'}
predicted_stages = [reverse_stage_mapping[num] for num in stage_preds]

# ============================================================
# DYNAMIC GOAL REGRESSION
# ============================================================
goals_col = 'total_goals_scored' if 'total_goals_scored' in train_df_working.columns else 'total_goals'
y_goals = train_df_working[goals_col].to_numpy()

rf_reg = RandomForestRegressor(n_estimators=150, max_depth=6, min_samples_leaf=2, random_state=42, n_jobs=-1)
rf_reg.fit(X_clean, y_goals)

predicted_goals = np.round(np.clip(rf_reg.predict(test_matrix), 0, None)).astype(int)

# ============================================================
# EXPORT
# ============================================================
submission = pd.read_csv("SampleSubmission.csv")
submission['total_goals'] = predicted_goals
submission['Target'] = predicted_stages
submission.to_csv("dynamic_spread_submission.csv", index=False)

print("📋 Sample breakdown of the new file:")
print(submission['Target'].value_counts())
print("\n🚀 Saved to 'dynamic_spread_submission.csv'. Ready to upload!")

⚡ Running Dynamic Distribution Pipeline...


/home/livvy/.local/lib/python3.14/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


📋 Sample breakdown of the new file:
Target
group    48
Name: count, dtype: int64

🚀 Saved to 'dynamic_spread_submission.csv'. Ready to upload!


In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

print("⚡ Running Dynamic Distribution Pipeline...")

train_df_working = train_data.copy()
test_df_working = pd.read_csv("Test.csv")

# 1. Host flags
hosts_2026 = ['USA', 'CAN', 'MEX', 'United States', 'Canada', 'Mexico']
test_df_working['is_host'] = 0
for col in ['team_id', 'team_code', 'team_name']:
    if col in test_df_working.columns:
        test_df_working['is_host'] = test_df_working[col].isin(hosts_2026).astype(int)

# 2. Re-introduce key performance metrics
for df in [train_df_working, test_df_working]:
    if 'goals_per_match' in df.columns:
        df['attack_defense_ratio'] = df['goals_per_match'] / (df['conceded_per_match'] + 1e-5)
        df['dominance_score'] = df['win_rate'] * (df['goal_difference'] / (df['matches_played'] + 1e-5))
        df['defensive_resilience'] = df['total_wins'] - df['total_goals_conceded']

clean_features = ['goals_per_match', 'conceded_per_match', 'win_rate', 'goal_difference', 'is_host', 'count_teams', 'year', 'attack_defense_ratio', 'dominance_score', 'defensive_resilience']

for col in clean_features:
    if col not in test_df_working.columns:
        test_df_working[col] = train_df_working[col].median()
    else:
        test_df_working[col] = test_df_working[col].fillna(train_df_working[col].median())
    train_df_working[col] = train_df_working[col].fillna(train_df_working[col].median())

X_clean = train_df_working[clean_features].to_numpy()
test_matrix = test_df_working[clean_features].to_numpy()

# ============================================================
# CLASSIFICATION WITH DYNAMIC SPREAD
# ============================================================
stage_mapping = {
    'group': 0, 'roundof32': 1, 'roundof16': 2, 
    'qf': 3, 'sf': 4, 'runnerup': 5, 'champion': 6
}
target_col = 'stage_target' if 'stage_target' in train_df_working.columns else 'stage_reached'
y_stage = train_df_working[target_col].map(stage_mapping).fillna(0).astype(int).to_numpy()

# Use balanced class weights to force predictions outside of just 'group'
lgb_stage = lgb.LGBMClassifier(
    objective='multiclass', 
    num_class=7, 
    n_estimators=150, 
    learning_rate=0.03, 
    max_depth=4,
    class_weight='balanced',
    random_state=42, 
    verbose=-1
)
lgb_stage.fit(X_clean, y_stage)

stage_preds = lgb_stage.predict(test_matrix)
reverse_stage_mapping = {0: 'group', 1: 'roundof32', 2: 'roundof16', 3: 'qf', 4: 'sf', 5: 'runnerup', 6: 'champion'}
predicted_stages = [reverse_stage_mapping[num] for num in stage_preds]

# ============================================================
# DYNAMIC GOAL REGRESSION
# ============================================================
goals_col = 'total_goals_scored' if 'total_goals_scored' in train_df_working.columns else 'total_goals'
y_goals = train_df_working[goals_col].to_numpy()

rf_reg = RandomForestRegressor(n_estimators=150, max_depth=6, min_samples_leaf=2, random_state=42, n_jobs=-1)
rf_reg.fit(X_clean, y_goals)

predicted_goals = np.round(np.clip(rf_reg.predict(test_matrix), 0, None)).astype(int)

# ============================================================
# EXPORT
# ============================================================
submission = pd.read_csv("SampleSubmission.csv")
submission['total_goals'] = predicted_goals
submission['Target'] = predicted_stages
submission.to_csv("dynamic_spread_submission.csv", index=False)

print("📋 Sample breakdown of the new file:")
print(submission['Target'].value_counts())
print("\n🚀 Saved to 'dynamic_spread_submission.csv'. Ready to upload!")

In [ ]:
🟢 Tier 1 — Most Relevant (Use These First)
DatasetWhy RelevantFeatures to Extractteam_appearancesGoals scored/conceded per matchgoals_for, goals_against, win/loss/drawqualified_teamsStage each team reached⭐ Your TARGET variablegroup_standingsGroup stage performancewins, losses, draws, goals_fortournament_standingsFinal rankings per tournamentHistorical finishing positionsmatchesFull match resultsHome/away performance, scoresgoalsIndividual goal dataGoals per game, scoring patterns

🟡 Tier 2 — Moderately Relevant (Use for Feature Engineering)
DatasetWhy RelevantFeatures to ExtracttournamentsTournament contextYear, format, number of teamssquadsSquad size, experienceNumber of players, avg ageplayersPlayer detailsExperience, agepenalty_kicksShootout performancePenalty win ratebookingsDisciplineCards per gamesubstitutionsSquad depthSubstitution patternshost_countriesHome advantageis_host flagaward_winnersStar playersGolden Boot, Golden Ball winners

🔴 Tier 3 — Less Relevant (Skip or Use Last)
DatasetWhy Less RelevantconfederationsJust reference datamanagersLimited predictive valuerefereesNot relevant to outcomesstadiumsNot relevant to outcomesmanager_appointmentsLimited predictive valuereferee_appointmentsNot relevantmanager_appearancesNot relevantreferee_appearancesNot relevantplayer_appearancesToo granulartournament_stagesJust structural infogroupsJust group namesawardsRedundant with award_winnersteamsJust reference data

Your Priority Order
1. team_appearances   ← Goals & match results
2. qualified_teams    ← Target variable (stage reached)
3. group_standings    ← Group performance features
4. matches            ← Win/loss/draw history
5. goals              ← Scoring patterns
6. host_countries     ← Home advantage flag
7. penalty_kicks      ← Penalty shootout record
8. squads/players     ← Squad experience & depth

Suggested Feature Engineering Plan
python# From team_appearances
avg_goals_scored    = goals scored per match historically
avg_goals_conceded  = goals conceded per match historically
win_rate            = wins / total matches

# From group_standings
avg_group_points    = average points in group stage
group_qualification_rate = times qualified from group

# From qualified_teams
best_stage_reached  = best ever stage historically
avg_stage_reached   = average stage across tournaments

# From host_countries
is_host             = 1 if hosting 2026, else 0

# From penalty_kicks
penalty_win_rate    = shootout wins / total shootouts